## Pre-processing Notebook

This notebook contains the instructions on how to process the data once it is downloaded, download instructions can be found [here](https://github.com/ppuentex/detection_comparison-CRB/blob/main/code/download_instructions.ipynb). In this notebook, you will find the following: 

1. Instruction on cropping the data for a given watershed boundary (ie. shapefile). 
2. Calculate the mode (most frequent pixels) for a given time period using Dask and Numba. 
3. Reprojecting raster data to have the same projection in meters, resolution of pixels is either 10m or 30m, and make sure nothing is shifted in the process.
4. Calculating the Kronecker product to get downscale Landsat 30m pixels to match the Sentinel 10m pixels, without having to reclassify. 

*Note: These instructions are written for the Colorado River Basin specifically but can be applied elsewhere with a new changes on area of interest when downloading.*

These instructions assume you have already combined the tiles through mosaicking them, it is recommended to use QGIS or use virtual merge through `gdalbuildvrt` [documentation here](https://gdal.org/en/stable/programs/gdalbuildvrt.html). 

Below is an example snippet to do it using Python. 

```
from pathlib import Path
import subprocess

#List of cropped tif files
folder_path = Path('./data')
vrt_path = folder_path / "merged.vrt"

tif_files = [str(tif) for tif in folder_path.glob('*_cropped_10m.tif')]
subprocess.run(["gdalbuildvrt", str(vrt_path)] + [str(file) for file in tif_files])

```
The code above creates the virtual .tif file and runs smoothly on your local machine. Next, this can be saved to your local machine in a compressed form, this does not lose any information. The code below saves your mosaic file that is necessary. 

```
import rasterio as rio
from rasterio.enums import Resampling

output_path = "./data/merged_land_cover_2019.tif"

with rio.open(vrt_path) as src:
    profile = src.profile.copy()
    profile.update({
        "driver": "GTiff",
        "compress": "LZW",
        "tiled": True,
        "blockxsize": 512,
        "blockysize": 512
    })

    with rio.open(output_path, "w", **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window=window, resampling=Resampling.nearest)
            dst.write(data, window=window, indexes=1)
```

### 1. Cropping the data for a given watershed boundary 
This technique was used to create all the raster files in this study. The cropped versions of the landsat yearly datasets for 2015 to 2021 can be found in `data/landsat-yearly/` and the cropped version of the sentinel data can be found in `data/zenodo-data`. 

In [ ]:
import subprocess

subprocess.run([
    "gdalwarp",
    "-cutline", "./data/shapefiles/crb_boundary.shp",
    "-crop_to_cutline",
    "-dstnodata", "0",
    "-co", "COMPRESS=LZW",
    "-co", "TILED=YES",
    "<input tif file path>", #input file 
    "<output tif file path>" #output file
])

### 2. Calculating the Mode using Dask and Numba

The Landsat file of the mode calculation from 2015 to 2021 can be found in `data/zenodo-data/mode_crb_landsat.tif`. Note that this is not the version that is used in the analysis due to still needing to reproject and resample to meet the Sentinel resolution. 

In [ ]:
from pathlib import Path 
import rioxarray
import numpy as np
import dask.array as da 
import numba 
from dask.diagnostics import ProgressBar
import rasterio

mode_output_file = '../data/zenodo-data/mode_landsat.tif'

folder_path = Path('../data/landsat-yearly')

tif_files = [str(tif_file) for tif_file in folder_path.glob('*.tif')]

tif_files #list of files that we will calculate the mode for 

['../data/landsat-yearly/2019_CRB.tif',
 '../data/landsat-yearly/2018_CRB.tif',
 '../data/landsat-yearly/2015_CRB.tif',
 '../data/landsat-yearly/2017_CRB.tif',
 '../data/landsat-yearly/2021_CRB.tif',
 '../data/landsat-yearly/2020_CRB.tif',
 '../data/landsat-yearly/2016_CRB.tif']

In [35]:
#stack the files using dask 
data_arrays = [rioxarray.open_rasterio(fp, masked = True).squeeze().chunk({"x": 1000, "y": 1000}) for fp in tif_files]
stacked_data = da.stack(data_arrays,axis=-1) #convert to dask array
stacked_data = stacked_data.astype(np.uint8)
stacked_data.shape

(53809, 40320, 7)

In [ ]:
#Using Numba to improve the efficiency of the mode calculation 
@numba.jit(nopython=True, parallel=True)
def numba_mode(arr):
    """Computes mode along the last axis using Numba."""
    h, w = arr.shape  # height, width 
    mode_result = np.zeros((h, w), dtype=np.uint8)
    
    for i in numba.prange(h):  # Iterate over rows
        for j in numba.prange(w):  # Iterate over cols
            values = arr[i,j,:].astype(np.uint8)
            # Count the occurrences using np.bincount
            counts = np.bincount(values)
            
            #find the mode (value with the max count)
            mode_result[i, j] = np.argmax(counts)  
    return mode_result


#Compute Mode Efficiently (should be much faster)
with ProgressBar():
    stacked_numpy = stacked_data.compute()
    mode_result = numba_mode(stacked_numpy)

In [ ]:
#save the array with the original metadata from the .tiff file 
with rasterio.open(tif_files[0]) as src:
    profile = src.meta.copy()
        
    #update metadata 
    profile.update({'count': 1}) #specify the number of bands
    profile.update({"compress": "lzw"}) 

    #write 
    with rasterio.open(mode_output_file, 'w', **profile) as dst:
        dst.write(mode_result, 1) #write data into first band